# Bachelor's thesis topic: Sentiment analysis based on multifractal methods

Sentiment analysis is crucial for decision-making in corporate platforms and for users. Traditional techniques face limitations, such as low prediction accuracy, large training datasets, and computational inefficiency. My project addresses these limitations by applying multifractal methods combined with machine learning techniques, reducing data features while preserving accuracy.








# Data Loading

The RuReviews corpus consisting of approximately 19,000 user reviews of the online women's clothing store was used as data.

In [ ]:
import numpy as np
import pandas as pd

full_data=pd.read_csv('RuReviews.csv',sep=',')

In [ ]:
full_data

## 1. Data preprocessing

In [ ]:
import nltk
!pip install pymystem3
nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt')
nltk.download('stopwords')

import numpy as np
import pandas as pd
from nltk.corpus import stopwords
import string
import re
from pymystem3 import Mystem

def remove_stopwords(i):
    stop_words = stopwords.words('russian')
    stop_words.remove('??')
    stop_words.remove('???')
    filtered_sentence = []
    for w in i:
        if w not in stop_words:
            filtered_sentence.append(w)
    return filtered_sentence

def data_preprocessing(data):
    num_steps = 12
    step = 1
    print('Step {}/{}: Drop duplicats and empty sentences '.format(step, num_steps))
    data.drop_duplicates(subset='Original_review',keep='first', inplace=True,)
    data.dropna(inplace=True)
    data=data.reset_index(drop=True)
    step += 1
    print('Step {}/{}: Converting to lower case'.format(step, num_steps))
    data['Review']= [i.lower() for i in data['Original_review']]

    step += 1
    print('Step {}/{}: Removing punctuation'.format(step, num_steps))
    data['Review'] = [ re.sub(r'[^\w\s]', ' ', i) for i in data['Review']]

    step += 1
    print('Step {}/{}: Removing single words'.format(step, num_steps))
    data['Review'] = [re.sub(r"\b[?-??-?]\b", " ", i) for i in data['Review']]

    step += 1
    print('Step {}/{}: Removing mentions and hashtag symbols'.format(step, num_steps))
    data['Review'] = [re.sub(r"@[a-zA-Z0-9]+|\#[a-zA-Z0-9]", " ", str(i)) for i in data['Review']]

    step += 1
    print('Step {}/{}: Removing URLs'.format(step, num_steps))
    data['Review'] = [re.sub(r'http\S+', " ", str(i)) for i in data['Review']]

    step += 1
    print('Step {}/{}: Removing numbers'.format(step, num_steps))
    data['Review'] = [re.sub(r"[0-9]", " ", i) for i in data['Review']]

    step += 1
    print('Step {}/{}: Removing latin letters'.format(step, num_steps))
    data['Review'] = [re.sub(r"[a-zA-Z]", " ", i) for i in data['Review']]

    step += 1
    print('Step {}/{}: Removing empty space'.format(step, num_steps))
    data['Review'] = [' '.join(i.split())for i in data['Review']]

    step += 1
    print('Step {}/{}: Lemmatization'.format(step, num_steps))
    mystem = Mystem()
    data['Review'] = [mystem.lemmatize(i) for i in data['Review']]

    step += 1
    print('Step {}/{}: Removing stop words'.format(step, num_steps))
    data['Review'] = [remove_stopwords(i) for i in data['Review']]
    step += 1

    print('Step {}/{}: Removing empty sentences'.format(step, num_steps))
    data = data[(data['Review'].str.len() != 0)]
    data['Review']=[' '.join(i) for i in data['Review']]
    data['Review'] = [' '.join(i.split())for i in data['Review']]
    data['Review'] = [i.replace("\n","") for i in data['Review']]
    data.drop_duplicates(subset='Review',keep='first',inplace=True)
    data['Review'] = [i.split() for i in data['Review']]
    return data


In [ ]:
prepr_data=data_preprocessing(full_data)

In [ ]:
prepr_data

Choose only 5000 positive and negative comments randomly

In [ ]:
from sklearn.utils import shuffle

prepr_data = shuffle(prepr_data)
positive=prepr_data[prepr_data.Sentiment==1]
negative=prepr_data[prepr_data.Sentiment==0]
min_value=5000
prepr_data =pd.concat([positive[:min_value],negative[:min_value]])
prepr_data = shuffle(prepr_data )
data=prepr_data.reset_index(drop=True)

In [ ]:
data

## 3. Words embedding

Navec is a library of pretrained word embeddings for russian language.

In [ ]:
!pip install navec

In [ ]:
!wget https://storage.yandexcloud.net/natasha-navec/packs/navec_hudlit_v1_12B_500K_300d_100q.tar

In [ ]:
from navec import Navec

path = 'navec_hudlit_v1_12B_500K_300d_100q.tar'
navec = Navec.load(path)
def emb(line):
    embedding=[navec[i] for i in line if i in navec]
    return embedding

embedding=[emb(i) for i in data.Review]

for i in range(len(embedding)):
  if embedding[i]==[]:
    print(i)
    embedding=embedding.pop(i)
    data=data.drop(labels=[i],axis = 0)

data=data.reset_index(drop=True)


In [ ]:
data

## 4. Sentences embedding

By averaging embedding vectors in a sentence, one can obtain a vector representation of the sentence that reflects its meaning.

In [ ]:
emb_mean=[]
emb_sent=[]
for i in range(len(embedding)):
  sum= np.zeros(len(embedding[i][0]))
  for j in  range(len(embedding[i])):
    sum=sum+embedding[i][j]
  sum/=len(embedding[i])
  emb_sent.append(sum)


In [ ]:
data['Embedding']=emb_sent

In [ ]:
data

# 5. Multifractal Detrended Fluctuation Analysis

Implementation of MF-DFA approach from scratch

In [ ]:
!pip install MFDFA
from MFDFA import MFDFA
import math
import matplotlib.pyplot as plt
import math

In [ ]:
s = np.arange(5,25,3)
order =3
result_dfa=[]
index=0
for word in emb_sent:
  h=[]
  q_=[1,2,3,4,5,6]
  for q in q_:
    y=[]
    lag, dfa = MFDFA(np.array(word), lag = s, q = q, order = order)
    dfa=dfa.flatten()
    dfa=[math.log(i) for i in dfa]
    y.append(dfa)
    x = [math.log(i) for i in lag]
    p1 = np.poly1d(np.polyfit(x, dfa, 1))
    h.append(p1.c[0])
  result_dfa.append(h)
  index+=1

In [ ]:
data['MF_DFA']=result_dfa

# Random Forest classification based on MF-DFA approach

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(data.MF_DFA, data.Sentiment, test_size=0.4, random_state=42, shuffle=True)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import tensorflow as tf
model = RandomForestClassifier()
print(len(X_train), len(X_test), len(y_train), len(y_test))
X_train=tf.keras.preprocessing.sequence.pad_sequences(X_train,dtype='float64',padding='post')
X_test=tf.keras.preprocessing.sequence.pad_sequences(X_test,dtype='float64',padding='post')
model.fit(X_train, y_train)
prediction_linear_rf = model.predict(X_test)
report = classification_report(y_test, prediction_linear_rf)
print(report)


# 5.  Multifractal detrended moving average analysis

Implementation of MF-DMA approach from scratch (Applying this approach might take a long time)

In [ ]:
import matplotlib.pyplot as plt
import math

def mf_dma(word,s=7, q=-2):
  y=[]
  sum=0
  for i in word:
    sum+=i
    y.append(sum)
  y_=[]
  E=[]
  for i in range(s,len(y)+1):
    sum=np.sum(y[i-s:i])
    temp=y[i-1]-sum/len(y[i-s:i])
    y_.append(sum/s)
    E.append(temp)
  I_s=math.floor(len(y_)/s)
  F_2=[]
  for i in range(s,len(E)+1,s):
    pow=np.power(E[i-s:i],2)
    sum=np.sum(pow)
    F_2.append(sum/s)
  for i in range(len(E),s-1,-s):
    pow=np.power(np.flip(E[i-s:i],0),2)
    sum=np.sum(pow)
    F_2.append(sum/s)
  if q!=0:
    sum=0
    for i in range(len(F_2)):
      pow=math.pow(F_2[i],(q/2))
      sum=sum+pow
    sum/=2*I_s
    F_q=math.pow(sum,(1/q))
  else:
    sum=0
    for i in range(len(F_2)):
      sum+=math.log(F_2[i])
    sum/=4*I_s
    F_q=math.exp(sum)
  return math.log(F_q)

In [ ]:
def hurst(word, q=-10):
  s = np.arange(3,13,1)
  x = [math.log(i) for i in s]
  y = [mf_dma(word,i,q) for i in s]
  p1 = np.poly1d(np.polyfit(x, y, 1))
  h.append(p1.c[0])

In [ ]:
result_dma=[]
index=0

for i in data.Embedding:
  h=[]
  q_=[0,1,2,3,4,5]
  for q in q_:
    hurst(i, q)
  result_dma.append(h)
  index+=1

In [ ]:
data['MF_DMA']=result_dma

# Random Forest classification based on MF-DMA approach

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(data.MF_DMA, data.Sentiment, test_size=0.4, random_state=42, shuffle=True)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import tensorflow as tf
model = RandomForestClassifier()
print(len(X_train), len(X_test), len(y_train), len(y_test))
X_train=tf.keras.preprocessing.sequence.pad_sequences(X_train,dtype='float64',padding='post')
X_test=tf.keras.preprocessing.sequence.pad_sequences(X_test,dtype='float64',padding='post')
model.fit(X_train, y_train)
prediction_linear_rf = model.predict(X_test)
report = classification_report(y_test, prediction_linear_rf)
print(report)


# Random Forest classification based on classical approach based on GloVe embedding

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(data.Embedding, data.Sentiment, test_size=0.4, random_state=42, shuffle=True)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import tensorflow as tf
model = RandomForestClassifier()
print(len(X_train), len(X_test), len(y_train), len(y_test))
X_train=tf.keras.preprocessing.sequence.pad_sequences(X_train,dtype='float64',padding='post')
X_test=tf.keras.preprocessing.sequence.pad_sequences(X_test,dtype='float64',padding='post')
model.fit(X_train, y_train)
prediction_linear_rf = model.predict(X_test)
report = classification_report(y_test, prediction_linear_rf)
print(report)
